# SAIR IC50 Oracle — Colab Training

Trains the IC50 oracle (ESM2 mean-pool protein encoder + GINEConv ligand encoder + RDKit descriptors) on Colab GPU.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free) or **A100** (Pro)
2. Make sure mean-pooled ESM embeddings exist in `data/esm_embeddings/`  
   (generated by `scripts/precompute_esm.py` — without `--residue-dir`)
3. Run all cells top-to-bottom
4. If the session disconnects: re-run cells 1–4, then set `RESUME = 'checkpoints/best.pt'` in cell 6

## Cell 1 — Verify GPU and mount Google Drive

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found — change runtime type to GPU before continuing."
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Install packages

PyTorch Geometric compiled extensions must match the exact PyTorch + CUDA version —
the URL is computed dynamically. If they fail, PyG falls back to pure-PyTorch scatter
ops: GINEConv still works correctly, just ~20% slower.

In [ ]:
import subprocess, sys, torch

torch_ver = torch.__version__.split('+')[0]
cuda_tag  = "cu" + torch.version.cuda.replace('.','')
pyg_url   = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"

print(f"PyTorch {torch_ver}  |  {cuda_tag}")
print(f"PyG wheel URL: {pyg_url}\n")

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

print("Installing torch_geometric...")
pip("torch_geometric")

print("Installing compiled PyG extensions (torch_scatter, torch_sparse)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "torch_scatter", "torch_sparse", "-f", pyg_url]
)
if result.returncode != 0:
    print("\u26a0  Compiled extensions unavailable — using pure-PyTorch fallback (correct, ~20% slower)")
else:
    print("\u2713  Compiled extensions installed")

print("Installing rdkit, psutil, wandb...")
pip("rdkit", "psutil", "wandb")

print("\n\u2713 All packages ready")

## Cell 3 — Set working directory and Python path

In [ ]:
import sys, os

PROJECT = '/content/drive/MyDrive/sair-ic50-oracle'
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

required = {
    'data/sair.parquet'                 : os.path.exists,
    'data/esm_embeddings/'              : os.path.isdir,
    'data/splits/train.txt'             : os.path.exists,
    'data/splits/val.txt'               : os.path.exists,
    'data/long_protein_annotations.csv' : os.path.exists,
    'config/default.yaml'               : os.path.exists,
}
optional = {
    'data/ligand_cache.pt': os.path.exists,
}

all_ok = True
for path, check in required.items():
    ok = check(path)
    print(f"{'✓' if ok else '✗'}  {path}")
    all_ok = all_ok and ok

for path, check in optional.items():
    ok = check(path)
    print(f"{'✓' if ok else '⚠'} (optional) {path}")
    if not ok:
        print(f"   → ligand_cache.pt missing: graphs/descriptors will be computed in-memory (~2–3 min slower)")

if not all_ok:
    raise RuntimeError("One or more required files are missing. See README Quickstart.")

from oracle.dataset import SAIRDataset
from oracle.model   import IC50Oracle
print("\n✓ Project imports OK")

## Cell 4 — Stage data to local SSD

Google Drive FUSE adds ~100 ms overhead **per file**. This cell copies everything
to Colab's local SSD (`/tmp/sair/`) once per session.

- `sair.parquet` — large sequential read benefits from SSD
- `esm_embeddings/` — 3,563 × .pt files (mean-pooled [1280] float32 tensors, ~17 MB)
- `ligand_cache.pt` — pre-computed graphs + RDKit descriptors for all unique SMILES  
  Generate once locally: `python scripts/precompute_ligands.py --parquet data/sair.parquet --output data/ligand_cache.pt`

Checkpoints, annotations, and splits paths are set to absolute Drive paths so they
persist across sessions even if this cell is re-run before Cell 3 sets the CWD.

**Re-run this cell at the start of every new session** (local SSD is wiped on disconnect).

In [ ]:
import shutil, os, time, yaml

SSD = '/tmp/sair'
os.makedirs(SSD, exist_ok=True)

items = [
    ('data/sair.parquet',   f'{SSD}/sair.parquet',   False, 'sair.parquet'),
    ('data/esm_embeddings', f'{SSD}/esm_embeddings',  True,  'ESM mean-pool embeddings'),
]
# Stage ligand cache only if it exists
if os.path.exists('data/ligand_cache.pt'):
    items.append(('data/ligand_cache.pt', f'{SSD}/ligand_cache.pt', False, 'ligand graph+descriptor cache'))

t0 = time.time()
for src, dst, is_dir, label in items:
    if os.path.exists(dst):
        print(f"  ✓ {label} — already on SSD, skipping")
        continue
    print(f"  Copying {label} ...", end='', flush=True)
    t = time.time()
    shutil.copytree(src, dst) if is_dir else shutil.copy2(src, dst)
    print(f" {time.time()-t:.0f}s")

with open('config/default.yaml') as f:
    cfg = yaml.safe_load(f)

# Fast paths: read large files from SSD
cfg['paths']['parquet']   = f'{SSD}/sair.parquet'
cfg['paths']['esm_cache'] = f'{SSD}/esm_embeddings/'
if os.path.exists(f'{SSD}/ligand_cache.pt'):
    cfg['paths']['ligand_cache'] = f'{SSD}/ligand_cache.pt'
else:
    cfg['paths']['ligand_cache'] = None  # fall back to in-memory computation

# Absolute Drive paths for outputs and small read-only files — these must resolve
# to Google Drive regardless of whether CWD was set correctly after a reconnect.
cfg['paths']['checkpoints'] = f'{PROJECT}/checkpoints'
cfg['paths']['annotations']  = f'{PROJECT}/data/long_protein_annotations.csv'
cfg['paths']['splits']       = f'{PROJECT}/data/splits/'

SESSION_CONFIG = f'{SSD}/colab_config.yaml'
with open(SESSION_CONFIG, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"\n✓ Data staged in {time.time()-t0:.0f}s")
print(f"  Parquet       : {SSD}/sair.parquet")
print(f"  ESM mean-pool : {SSD}/esm_embeddings/")
print(f"  Ligand cache  : {SSD}/ligand_cache.pt" if os.path.exists(f'{SSD}/ligand_cache.pt') else "  Ligand cache  : not found — will compute in-memory")
print(f"  Config        : {SESSION_CONFIG}")
print(f"  Checkpoints   : {cfg['paths']['checkpoints']} (Google Drive — persistent)")

## Cell 5 — (Optional) wandb login

Skip this cell to run without wandb logging.  
To enable: paste your API key when prompted, or pre-set the `WANDB_API_KEY` env var.

In [ ]:
import wandb
wandb.login()

## Cell 6 — Train

Calls `train()` directly so `tqdm.auto` renders a live progress bar widget.

Dataset init takes ~10–20 s (mean-pool embeddings are ~3.7 GB, loaded once from SSD).  
Each epoch shows a tqdm bar with live loss and lr.

**After a disconnection:** re-run cells 1–4, then set `RESUME = 'checkpoints/best.pt'`.

In [ ]:
import yaml
from scripts.train import train, load_config

SESSION_CONFIG = '/tmp/sair/colab_config.yaml'
RESUME         = None   # set to 'checkpoints/best.pt' to resume after disconnect

config = load_config(SESSION_CONFIG)
train(config, resume=RESUME)

## Cell 7 — Checkpoint inspection

Run after training completes (or any time) to see the best checkpoint metrics.

In [ ]:
import torch

ckpt = torch.load('checkpoints/best.pt', map_location='cpu', weights_only=False)
print(f"Best epoch              : {ckpt['epoch']}")
print(f"Per-prot Spearman (best): {ckpt['best_spearman']:.4f}")
print(f"Val Spearman            : {ckpt['val_spearman']:.4f}")
print(f"Val MAE                 : {ckpt['val_mae']:.4f}")
print(f"Global step             : {ckpt['global_step']:,}")
print(f"Epochs w/o improvement  : {ckpt.get('epochs_without_improvement', '?')}")

---
## Reference: expected timings

| Phase | T4 (free) | A100 (Pro) |
|---|---|---|
| Data staging — Cell 4 (once per session) | ~60–90 s | ~60–90 s |
| Dataset init (mean-pool load + graph cache) | ~20 s | ~20 s |
| Training per epoch | ~3–4 min | ~1–2 min |
| Full run with patience=15 (~30 epochs) | ~2 hrs | ~45 min |

## Reference: reconnection checklist

After a session disconnects, `/tmp/` is wiped and packages are uninstalled. On reconnect:

1. Cell 1 — GPU check + Drive mount
2. Cell 2 — reinstall packages
3. Cell 3 — set paths + imports
4. Cell 4 — re-stage data to SSD
5. Cell 6 — set `RESUME = 'checkpoints/best.pt'` and run

Checkpoints survive in Google Drive.